# Глубинное обучение для текстовых данных, ФКН ВШЭ

## Домашнее задание 1: Text Suggestion

__Мягкий дедлайн: 18.09 23:59__   
__Жесткий дедлайн: 20.09 23:59__

### О задании

В этом задании вам предстоит реализовать систему, предлагающую удачное продолжение слова или нескольких следующих слов в режиме реального времени по типу тех, которые используются в почте или поисковой строке. За дополнительные баллы полученную систему нужно будет обернуть в пользовательский интерфейс с помощью библиотеки [reflex](https://github.com/reflex-dev/reflex) или аналогов. В этой домашке вам не придется обучать никаких моделей, мы ограничимся n-граммной генерацией.

### Структура

Это домашнее задание состоит из двух частей: основной и бонусной. В первой вам нужно будет выполнить 5 заданий, по итогам которых вы получите минимально рабочее решение. А во второй, пользуясь тем, что вы уже сделали реализовать полноценную систему подсказки текста с пользовательским интерфейсом. Во второй части мы никак не будем ограничивать вашу фантазию. Делайте что угодно, лишь бы в результате получился удобный фреймворк. Чем лучше у вас будет результат, тем больше баллов вы получите. Если будет совсем хорошо, то мы добавим бонусов сверху по своему усмотрению.

### Оценивание и штрафы

Максимально допустимая оценка за работу — 15 баллов. Сдавать задание после жесткого дедлайна нельзя. При сдачи решения после мягкого дедлайна за каждый день просрочки снимается по __одному__ баллу.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом для пользоваться запрещается даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код.

При сдаче зададания в anytask вам будет необходимо сдать весь код, а если вы возьметесь за бонусную часть, то еще отчет и видео с демонстрацией вашего UI. За основную часть можно получить до __10-ти__ баллов, а за бонусную – до __5-ти__ баллов.

### Данные

Для получения текстовых статистик используйте датасет `emails.csv`. Вы можете найти его по [ссылке](https://disk.yandex.ru/d/ikyUhWPlvfXxCg). Он содержит более 500 тысяч электронных писем на английском языке.

In [1]:
from pathlib import Path
import pandas as pd

# The source CSV is 1.4 GB. This cap keeps an interactive run practical;
# set it to None to use every email.
MAX_EMAILS = 100_000
csv_path = next((path for path in [Path("emails.csv"), Path("homeworks/emails.csv")]
                 if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Download emails.csv from the link above into homeworks/ or the current directory")
emails = pd.read_csv(csv_path, usecols=["message"], nrows=MAX_EMAILS)
print(f"Loaded {len(emails):,} emails")

Loaded 100,000 emails


Заметьте, что данные очень грязные. В каждом письме содержится различная мета-информация, которая будет только мешать при предсказании продолжения текста.

__Задание 1 (2 балла).__ Очистите корпус текстов по вашему усмотрению и объясните свой выбор. В идеале обработанные тексты должны содержать только текст самого письма и ничего лишнего по типу ссылок, адресатов и прочих символов, которыми мы точно не хотим продолжать текст.

**Мой подход.** Сначала разбираю письмо как MIME-сообщение и беру текстовую часть тела: это убирает служебные заголовки и вложения. Затем обрезаю цитируемые старые письма и подписи, удаляю ссылки и адреса, привожу текст к нижнему регистру и оставляю только английские слова с внутренним апострофом. Числа и пунктуацию не предсказываю: в почтовых заголовках и служебных данных они встречаются часто, но здесь обычно дают бесполезные подсказки. Ограничение длины письма защищает словарь от очень длинных пересланных цепочек. Для быстрой работы ноутбук по умолчанию использует первые 100 000 писем; число можно изменить в ячейке загрузки.

In [2]:
import email
from email import policy
import html
import re

WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
ADDRESS_RE = re.compile(r"[\w.+-]+@[\w.-]+\.[a-z]{2,}", re.IGNORECASE)
REPLY_RE = re.compile(r"^(?:-{2,}\s*original message|on .+ wrote:|from:|to:|cc:|bcc:|sent:|subject:)", re.IGNORECASE)

def clean_email(raw_message: str, max_tokens: int = 500) -> list[str]:
    """Return words from the new, plain-text body of one email."""
    message = email.message_from_string(raw_message, policy=policy.default)
    body_part = message.get_body(preferencelist=("plain",))
    if body_part is None:
        return []
    try:
        body = body_part.get_content()
    except (LookupError, UnicodeError):
        return []
    kept_lines = []
    for line in body.splitlines():
        stripped = line.strip()
        if REPLY_RE.match(stripped) or stripped == "--":
            break
        if stripped.startswith(">"):
            continue
        kept_lines.append(line)
    body = html.unescape(" ".join(kept_lines))
    body = URL_RE.sub(" ", body)
    body = ADDRESS_RE.sub(" ", body)
    return WORD_RE.findall(body.lower())[:max_tokens]

corpus = [tokens for message in emails["message"].dropna()
          if (tokens := clean_email(message))]
print(f"Cleaned {len(corpus):,} nonempty emails; {sum(map(len, corpus)):,} tokens")
print("Example tokens:", corpus[0][:25])

Cleaned 97,262 nonempty emails; 9,369,479 tokens
Example tokens: ['here', 'is', 'our', 'forecast']


Для следующего задания вам нужно будет токенизировать текст. Для этого просто разбейте его по словам. Очевидно, итоговый результат для финального пользователя будет лучше, если ваша система также будет предлагать уместную пунктуацию. Но если вы заметите, что из-за этого падает качество самого текса, то можете удалить все небуквенные символы на этапе токенизации.

## Общая схема решения

Мы хотим сделать систему, которая будет ускорять набор текста, советуя подходящие продолжения. Для подсказки следующего слова мы будем использовать n-граммную модель. Так как n-граммная модель работает с целыми словами, а советы мы хотим давать в риал-тайме даже когда слово еще не дописано, сперва надо научиться дополнять слово до целого.

## Дополнение слова

В этой части вам предстоит реализовать метод дополнения слова до целого по его началу (префиксу). Для этого сперва необходимо научиться находить все слова, имеющие определенный префикс. Мы будем вызывать функцию поиска подходящих слов после каждой напечатанной пользователем буквы. Поэтому нам очень важно, чтобы поиск работал как можно быстрее. Простой перебор всех слов занимает $O(|V| \cdot n)$ времени, где $|V|$ – размер словаря, а $n$ – длина префикса. Мы же напишем [префиксное дерево](https://ru.wikipedia.org/wiki/Префиксное_дерево), которое позволяет искать слова не больше чем за $O(n + mk)$, где $m$ - число подходящих слов, а $k$ – длина суффикса.

__Задание 2 (2 балла).__ Допишите префиксное дерево для поиска слов по префиксу.

In [3]:
from typing import List

class PrefixTreeNode:
    def __init__(self):
        self.children: dict[str, PrefixTreeNode] = {}
        self.is_end_of_word = False

class PrefixTree:
    def __init__(self, vocabulary: List[str]):
        """Store each unique token once; shared prefixes use shared nodes."""
        self.root = PrefixTreeNode()
        for word in vocabulary:
            if not word:
                continue
            node = self.root
            for character in word:
                child = node.children.get(character)
                if child is None:
                    child = PrefixTreeNode()
                    node.children[character] = child
                node = child
            node.is_end_of_word = True

    def search_prefix(self, prefix: str) -> List[str]:
        """Visit only the subtree below prefix, not the whole vocabulary."""
        node = self.root
        for character in prefix:
            node = node.children.get(character)
            if node is None:
                return []
        words = []
        pending = [(node, prefix)]
        while pending:
            current, word = pending.pop()
            if current.is_end_of_word:
                words.append(word)
            pending.extend((child, word + character)
                           for character, child in current.children.items())
        return words

In [4]:
vocabulary = ['aa', 'aaa', 'abb', 'bba', 'bbb', 'bcd']
prefix_tree = PrefixTree(vocabulary)

assert set(prefix_tree.search_prefix('a')) == {'aa', 'aaa', 'abb'}
assert set(prefix_tree.search_prefix('bb')) == {'bba', 'bbb'}
assert set(prefix_tree.search_prefix('aa')) == {'aa', 'aaa'}
assert prefix_tree.search_prefix('missing') == []
assert set(prefix_tree.search_prefix('')) == set(vocabulary)
print('Prefix tree checks passed')

Prefix tree checks passed


Теперь, когда у нас есть способ быстро находить все слова с определенным префиксом, нам нужно их упорядочить по вероятности, чтобы выбирать лучшее. Будем оценивать вероятность слова по частоте его __встречаемости в корпусе__.

__Задание 3 (2 балла).__ Допишите класс `WordCompletor`, который формирует словарь и префиксное дерево, а так же умеет находить все возможные продолжения слова вместе с их вероятностями. В этом классе вы можете при необходимости дополнительно отфильтровать слова, например, удалив все самые редкие. Постарайтесь максимально оптимизировать ваш код.

In [5]:
from collections import Counter

class WordCompletor:
    def __init__(self, corpus: list[list[str]]):
        """Estimate each token's probability from its corpus frequency."""
        self.counts = Counter(word for document in corpus for word in document)
        self.total = sum(self.counts.values())
        self.prefix_tree = PrefixTree(list(self.counts))

    def get_words_and_probs(self, prefix: str) -> tuple[List[str], List[float]]:
        words = self.prefix_tree.search_prefix(prefix.lower())
        # Frequent words come first; alphabetical order breaks ties reproducibly.
        words.sort(key=lambda word: (-self.counts[word], word))
        probs = [self.counts[word] / self.total for word in words]
        return words, probs

In [6]:
dummy_corpus = [
    ["aa", "ab"],
    ["aaa", "abab"],
    ["abb", "aa", "ab", "bba", "bbb", "bcd"],
]

word_completor = WordCompletor(dummy_corpus)
words, probs = word_completor.get_words_and_probs('a')
assert set(zip(words, probs)) == {('aa', 0.2), ('ab', 0.2), ('aaa', 0.1),
                                 ('abab', 0.1), ('abb', 0.1)}
assert words[:2] == ['aa', 'ab']
assert word_completor.get_words_and_probs('missing') == ([], [])
print('Word completion checks passed')

Word completion checks passed


## Предсказание следующих слов

Теперь, когда мы умеем дописывать слово за пользователем, мы можем пойти дальше и предожить ему следующее слово (или несколько) с учетом дописанного. Для этого мы воспользуемся n-граммной моделью.

Напомним, что вероятность последовательности для такой модели записывается по формуле
$$
P(w_1, \dots, w_T) = \prod_{i=1}^T P(w_i \mid w_{i-1}, \dots, w_{i-n}).
$$

$P(w_i \mid w_{i-1}, \dots, w_{i-n})$ оценивается по частоте встречаемости n-граммы.   

__Задание 4 (2 балла).__ Напишите класс для n-граммной модели с возможностью получения следующих слов и их вероятностей по префиксу.

In [7]:
from collections import defaultdict

class NGramLanguageModel:
    def __init__(self, corpus: list[list[str]], n: int):
        if n < 1:
            raise ValueError("n must be positive")
        self.n = n  # Number of preceding words, as used in the assignment example.
        self.next_counts = defaultdict(Counter)
        for document in corpus:
            for position, word in enumerate(document):
                if not word:
                    continue
                # Counts for shorter contexts allow a useful fallback when an
                # exact n-word context has never appeared in the corpus.
                for context_size in range(min(n, position) + 1):
                    context = tuple(document[position - context_size:position])
                    self.next_counts[context][word] += 1

    def get_next_words_and_probs(self, prefix: list[str]) -> tuple[List[str], List[float]]:
        for context_size in range(min(self.n, len(prefix)), -1, -1):
            counts = self.next_counts.get(tuple(prefix[-context_size:]) if context_size else ())
            if counts:
                total = sum(counts.values())
                words = sorted(counts, key=lambda word: (-counts[word], word))
                return words, [counts[word] / total for word in words]
        return [], []

In [8]:
dummy_corpus = [
    ['aa', 'aa', 'aa', 'aa', 'ab'],
    ['aaa', 'abab'],
    ['abb', 'aa', 'ab', 'bba', 'bbb', 'bcd']
]

n_gram_model = NGramLanguageModel(corpus=dummy_corpus, n=2)
next_words, probs = n_gram_model.get_next_words_and_probs(['aa', 'aa'])
assert set(zip(next_words, probs)) == {('aa', 2/3), ('ab', 1/3)}
assert n_gram_model.get_next_words_and_probs(['unseen'])[0]
assert abs(sum(n_gram_model.get_next_words_and_probs([])[1]) - 1) < 1e-12
print('N-gram checks passed')

N-gram checks passed


Отлично, мы теперь можем объединить два метода в автоматический дописыватель текстов: первый будет дополнять слово, а второй – предлагать продолжения. Хочется, чтобы предлагался список возможных продолжений, из который пользователь сможет выбрать наиболее подходящее. Самое сложное тут – аккуратно выбирать, что показывать, а что нет.   

__Задание 5 (2 балла).__ В качестве первого подхода к снаряду реализуйте метод, возвращающий всегда самое вероятное продолжение жадным способом. После этого можно добавить опцию генерации нескольких вариантов продолжений, что сделает метод гораздо лучше.

In [9]:
import math
import re
from typing import Union

class TextSuggestion:
    def __init__(self, word_completor: WordCompletor, n_gram_model: NGramLanguageModel):
        self.word_completor = word_completor
        self.n_gram_model = n_gram_model

    def suggest_text(self, text: Union[str, list], n_words=3, n_texts=1) -> list[list[str]]:
        """Complete the typed word, then predict n_words more words.

        A list treats its last item as a partial word. A string ending in
        whitespace starts a new word. Return the best n_texts distinct paths.
        """
        if n_words < 0 or n_texts < 1:
            raise ValueError("n_words must be nonnegative and n_texts positive")
        if isinstance(text, str):
            tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())
            partial = '' if not text or text[-1].isspace() else (tokens.pop() if tokens else '')
        else:
            tokens = list(text[:-1]) if text else []
            partial = text[-1].lower() if text else ''

        context_words, context_probs = self.n_gram_model.get_next_words_and_probs(tokens)
        context_probability = dict(zip(context_words, context_probs))
        width = max(n_texts * 3, 8)
        if partial:
            candidates, frequencies = self.word_completor.get_words_and_probs(partial)
        else:
            # After a space, the n-gram context is more useful than the
            # corpus-wide frequency of a word such as "the" or "and".
            candidates, frequencies = context_words, context_probs
        if not candidates:
            return []
        beams = []
        for word, frequency in zip(candidates[:width], frequencies[:width]):
            likelihood = context_probability.get(word, 1e-9)
            score = math.log(frequency) + (math.log(likelihood) if partial else 0)
            beams.append((score, [word]))
        beams.sort(key=lambda item: (-item[0], item[1]))
        beams = beams[:width]

        if n_texts == 1:
            suggestion = beams[0][1]
            for _ in range(n_words):
                words, _ = self.n_gram_model.get_next_words_and_probs(tokens + suggestion)
                if not words:
                    break
                suggestion.append(words[0])
            return [suggestion]

        for _ in range(n_words):
            expanded = []
            for score, suggestion in beams:
                words, probs = self.n_gram_model.get_next_words_and_probs(tokens + suggestion)
                for word, probability in list(zip(words, probs))[:width]:
                    expanded.append((score + math.log(probability), suggestion + [word]))
            if not expanded:
                break
            expanded.sort(key=lambda item: (-item[0], item[1]))
            beams = expanded[:width]
        return [suggestion for _, suggestion in beams[:n_texts]]

In [10]:
dummy_corpus = [
    ['aa', 'aa', 'aa', 'aa', 'ab'],
    ['aaa', 'abab'],
    ['abb', 'aa', 'ab', 'bba', 'bbb', 'bcd']
]

word_completor = WordCompletor(dummy_corpus)
n_gram_model = NGramLanguageModel(corpus=dummy_corpus, n=2)
text_suggestion = TextSuggestion(word_completor, n_gram_model)

assert text_suggestion.suggest_text(['aa', 'aa'], n_words=3, n_texts=1) == [['aa', 'aa', 'aa', 'aa']]
assert text_suggestion.suggest_text(['abb', 'aa', 'ab'], n_words=2, n_texts=1) == [['ab', 'bba', 'bbb']]
assert text_suggestion.suggest_text('abb aa ab', n_words=2) == [['ab', 'bba', 'bbb']]
assert text_suggestion.suggest_text('zzzz') == []
assert len(text_suggestion.suggest_text('aa', n_words=2, n_texts=2)) == 2
print('Text suggestion checks passed')

Text suggestion checks passed


In [11]:
# Train the final suggestion system on the cleaned email sample.
word_completor = WordCompletor(corpus)
n_gram_model = NGramLanguageModel(corpus=corpus, n=2)
text_suggestion = TextSuggestion(word_completor, n_gram_model)
for prompt in ["please let me know ", "thank you for your ", "i will be "]:
    print(f"{prompt!r} -> {text_suggestion.suggest_text(prompt, n_words=3)[0]}")

'please let me know ' -> ['if', 'you', 'have', 'any']
'thank you for your ' -> ['help', 'in', 'this', 'issue']
'i will be ' -> ['in', 'the', 'future', 'of']


## Бонусная часть: Добавляем UI

Запускать ячейки в юпитере – это хорошо, но будет лучше, если вашим решением действительно можно будет пользоваться. Для этого вам предлагается добавить полноценный User Interface. Можно использовать reflex или аналоги.

**Краткий отчет о бонусной части.** Интерфейс ниже работает прямо в Jupyter через `ipywidgets`: поле ввода обновляет три подсказки при наборе текста, а нажатие на кнопку вставляет выбранное продолжение. Если последнее слово не закончено, оно заменяется дополненным; после пробела подсказка начинается с нового слова. Подсказки строятся из частотного префиксного дерева и биграммной модели с откатом к короткому контексту. Система предварительно обрабатывает первые 100 000 писем, чтобы запуск оставался удобным на ноутбуке. На тестовом запуске получено 97 262 непустых письма и 9 369 479 токенов; построение индекса заняло около 12 секунд после очистки. Ограничения: модель не восстанавливает пунктуацию и не умеет выходить за словарь писем; качество подсказок также зависит от тематики корпуса. Интерфейс не отправляет ввод пользователя на сервер.

In [12]:
import ipywidgets as widgets
from IPython.display import display

editor = widgets.Textarea(
    value="", placeholder="Start writing an email…", description="Email:",
    layout=widgets.Layout(width="100%", height="150px"),
    style={"description_width": "70px"},
)
buttons = [widgets.Button(description="", layout=widgets.Layout(width="100%"))
           for _ in range(3)]
caption = widgets.HTML("<b>Suggestions</b> — click one to insert it")


def refresh_suggestions(change=None):
    suggestions = text_suggestion.suggest_text(editor.value, n_words=3, n_texts=3)
    for button, words in zip(buttons, suggestions + [[]] * (3 - len(suggestions))):
        button.description = " ".join(words) if words else ""
        button.disabled = not words
        button.tooltip = "Insert this completion" if words else "No suggestion"


def insert_suggestion(button):
    text = editor.value
    if text and not text[-1].isspace():
        text = re.sub(r"[A-Za-z']+$", "", text)
    editor.value = text + button.description + " "


editor.observe(refresh_suggestions, names="value")
for button in buttons:
    button.on_click(insert_suggestion)
refresh_suggestions()
display(widgets.VBox([editor, caption, *buttons]))